In [1]:

import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('dirty_cafe_sales.csv')
df_raw = df.copy()
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [3]:
print("Shape (rows, columns):", df.shape)
print("\nData types:")
print(df.dtypes)

Shape (rows, columns): (10000, 8)

Data types:
Transaction ID      object
Item                object
Quantity            object
Price Per Unit      object
Total Spent         object
Payment Method      object
Location            object
Transaction Date    object
dtype: object


In [4]:
null_report = pd.DataFrame({
    'Null Count': df.isnull().sum(),
    'Null %': (df.isnull().sum() / len(df) * 100).round(2)
})
print(null_report)

                  Null Count  Null %
Transaction ID             0    0.00
Item                     333    3.33
Quantity                 138    1.38
Price Per Unit           179    1.79
Total Spent              173    1.73
Payment Method          2579   25.79
Location                3265   32.65
Transaction Date         159    1.59


In [5]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [6]:
for col in df.select_dtypes(include='object').columns:
    print(f"\n{col} ({df[col].nunique()} unique):")
    print(df[col].value_counts(dropna=False).head(10))


Transaction ID (10000 unique):
Transaction ID
TXN_9226047    1
TXN_8567525    1
TXN_4583012    1
TXN_6796890    1
TXN_9933628    1
TXN_4302199    1
TXN_5548914    1
TXN_3528020    1
TXN_9668108    1
TXN_8076061    1
Name: count, dtype: int64

Item (10 unique):
Item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
UNKNOWN      344
NaN          333
Name: count, dtype: int64

Quantity (7 unique):
Quantity
5          2013
2          1974
4          1863
3          1849
1          1822
UNKNOWN     171
ERROR       170
NaN         138
Name: count, dtype: int64

Price Per Unit (8 unique):
Price Per Unit
3.0        2429
4.0        2331
2.0        1227
5.0        1204
1.0        1143
1.5        1133
ERROR       190
NaN         179
UNKNOWN     164
Name: count, dtype: int64

Total Spent (19 unique):
Total Spent
6.0     979
12.0    939
3.0     930
4.0     923
20.0    746
15.0    734
8.0     677
10.0    524
2.0  

In [7]:
df.describe()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_9226047,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


## Data Quality Report

The dataset contains [X] rows and [Y] columns. Missing values were found 
in the following columns: [column names with counts and percentages]. 
There are [N] fully duplicated rows. Data type issues: [e.g., Quantity 
and Price are stored as object instead of numeric because they contain 
text placeholders such as "ERROR" and "UNKNOWN"]. Value anomalies: 
[e.g., inconsistent capitalisation in categorical columns, placeholder 
values, unrealistic min/max values]. These issues will be addressed in 
the following sections.

In [8]:
print(df.columns.tolist())
print("\nNulls before this step:")
print(df.isnull().sum())

['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date']

Nulls before this step:
Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64


In [9]:
df = df.replace(['ERROR', 'UNKNOWN', 'N/A', ''], np.nan)
print(df.isnull().sum())

Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64


In [10]:
num_cols = ['Quantity', 'Price Per Unit', 'Total Spent']
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
print(df[num_cols].dtypes)

Quantity          float64
Price Per Unit    float64
Total Spent       float64
dtype: object


In [11]:
df['Total Spent'] = df['Total Spent'].fillna(df['Quantity'] * df['Price Per Unit'])
print("Total Spent nulls left:", df['Total Spent'].isnull().sum())

Total Spent nulls left: 40


In [12]:
for col in num_cols:
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)
    print(f"{col}: filled with median = {median_value}")

Quantity: filled with median = 3.0
Price Per Unit: filled with median = 3.0
Total Spent: filled with median = 8.0


In [13]:
cat_cols = ['Item', 'Payment Method', 'Location']
for col in cat_cols:
    df[col] = df[col].fillna('Unknown')

In [14]:
rows_before = len(df)
df = df.dropna(subset=['Transaction ID', 'Transaction Date'])
rows_removed = rows_before - len(df)
print("Rows removed:", rows_removed)
print(f"Share of dataset removed: {rows_removed / rows_before * 100:.2f}%")

Rows removed: 460
Share of dataset removed: 4.60%


In [15]:
print(df.isnull().sum())
print("Total nulls:", df.isnull().sum().sum())

Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64
Total nulls: 0


## Missing Data Handling

Text placeholders ("ERROR", "UNKNOWN") were first converted to proper 
missing values so that all gaps could be handled consistently. The 
strategy for each column was chosen as follows:

- **Quantity, Price Per Unit (numeric):** filled with the median. The 
median is robust to outliers, so extreme values do not distort the 
imputed number. Missing share: [X%] and [X%].
- **Total Spent (numeric):** recalculated as Quantity × Price Per Unit 
wherever both values were known, since this gives the exact value 
rather than an estimate. Any remaining gaps were filled with the 
median. Recalculated: [N] values.
- **Item, Payment Method, Location (categorical):** filled with 
"Unknown". Mode imputation would assign the most frequent category to 
customers whose real value is not known, which would bias the git 
distribution. Missing share: [X%], [X%], [X%].
- **Transaction ID, Transaction Date:** rows with a missing value were 
deleted ([N] rows, [X%] of the dataset), because an identifier or a 
date cannot be reliably imputed, and the share removed is small.

After these steps, the dataset contains no missing values.

In [16]:
full_duplicates = df.duplicated().sum()
print("Fully duplicated rows:", full_duplicates)

Fully duplicated rows: 0


In [17]:
id_duplicates = df.duplicated(subset=['Transaction ID']).sum()
print("Rows with a repeated Transaction ID:", id_duplicates)

Rows with a repeated Transaction ID: 0


In [18]:
df[df.duplicated(subset=['Transaction ID'], keep=False)].sort_values('Transaction ID').head(10)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date


In [19]:
rows_before = len(df)
df = df.drop_duplicates(subset=['Transaction ID'], keep='first')
rows_removed = rows_before - len(df)

print("Rows before:", rows_before)
print("Rows after:", len(df))
print("Duplicate rows removed:", rows_removed)

Rows before: 9540
Rows after: 9540
Duplicate rows removed: 0


In [20]:
print("Duplicates left:", df.duplicated(subset=['Transaction ID']).sum())
df.reset_index(drop=True, inplace=True)

Duplicates left: 0


## Duplicate Removal

The dataset was checked for fully identical rows and for repeated 
Transaction IDs. No duplicates were found (0 rows removed), so the 
dataset keeps all 9,540 rows. The check is retained in the notebook to 
document that this quality issue was verified and not skipped.

In [21]:
for col in ['Item', 'Payment Method', 'Location']:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))
print("\nTransaction Date dtype before:", df['Transaction Date'].dtype)
print(df['Transaction Date'].head())


Item:
Item
Juice       1124
Coffee      1123
Salad       1099
Cake        1082
Sandwich    1075
Smoothie    1048
Cookie      1035
Tea         1027
Unknown      927
Name: count, dtype: int64

Payment Method:
Payment Method
Unknown           3015
Digital Wallet    2197
Credit Card       2170
Cash              2158
Name: count, dtype: int64

Location:
Location
Unknown     3779
Takeaway    2889
In-store    2872
Name: count, dtype: int64

Transaction Date dtype before: object
0    2023-09-08
1    2023-05-16
2    2023-07-19
3    2023-04-27
4    2023-06-11
Name: Transaction Date, dtype: object


In [22]:
text_cols = ['Item', 'Payment Method', 'Location']
for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.title()

In [23]:
replacements = {
    'Payment Method': {'Cc': 'Credit Card', 'Cash ': 'Cash'},
    'Location': {'Instore': 'In-Store', 'Take Away': 'Takeaway'},
}
for col, mapping in replacements.items():
    df[col] = df[col].replace(mapping)

In [24]:
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')
print("Dtype after:", df['Transaction Date'].dtype)
print("Dates that could not be parsed:", df['Transaction Date'].isnull().sum())

Dtype after: datetime64[ns]
Dates that could not be parsed: 0


In [25]:
for col in text_cols:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))
df.dtypes


Item:
Item
Juice       1124
Coffee      1123
Salad       1099
Cake        1082
Sandwich    1075
Smoothie    1048
Cookie      1035
Tea         1027
Unknown      927
Name: count, dtype: int64

Payment Method:
Payment Method
Unknown           3015
Digital Wallet    2197
Credit Card       2170
Cash              2158
Name: count, dtype: int64

Location:
Location
Unknown     3779
Takeaway    2889
In-Store    2872
Name: count, dtype: int64


Transaction ID              object
Item                        object
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method              object
Location                    object
Transaction Date    datetime64[ns]
dtype: object

## Standardisation

Formatting consistency was checked in the categorical columns and the 
date column.

- **Item, Payment Method, Location:** leading and trailing whitespace 
was trimmed and capitalisation was unified (title case). The value 
counts before and after were identical, so no spelling or case variants 
were present in the source data (for example, Item contains 9 clean 
categories and Location contains only "Takeaway", "In-Store" and 
"Unknown"). The step is kept as a safeguard and to document that the 
check was performed.
- **Transaction Date:** converted from text (object) to datetime so 
that dates can be sorted, filtered and grouped by period. All values 
were parsed successfully (0 unparsed dates).

In [26]:
num_cols = ['Quantity', 'Price Per Unit', 'Total Spent']
summary = []

for col in num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    summary.append({
        'Column': col,
        'Q1': q1, 'Q3': q3, 'IQR': iqr,
        'Lower bound': lower, 'Upper bound': upper,
        'Outliers': n_outliers,
        'Outliers %': round(n_outliers / len(df) * 100, 2)
    })

pd.DataFrame(summary)

,Column,Q1,Q3,IQR,Lower bound,Upper bound,Outliers,Outliers %
0,Quantity,2.0,4.0,2.0,-1.0,7.0,0,0.00
1,Price Per Unit,2.0,4.0,2.0,-1.0,7.0,0,0.00
2,Total Spent,4.0,12.0,8.0,-8.0,24.0,259,2.71


In [27]:
df[num_cols].describe()

,Quantity,Price Per Unit,Total Spent
count,9540.000000,9540.000000,9540.000000
mean,3.024109,2.952673,8.920178
std,1.384186,1.243449,5.993427
min,1.000000,1.000000,1.000000
25%,2.000000,2.000000,4.000000
50%,3.000000,3.000000,8.000000
75%,4.000000,4.000000,12.000000
max,5.000000,5.000000,25.000000


In [28]:
col = 'Total Spent'
q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
iqr = q3 - q1
mask = (df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)
df[mask].sort_values(col, ascending=False).head(10)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
9513,TXN_6120851,Salad,5.0,5.0,25.0,Cash,Takeaway,2023-02-04
10,TXN_2548360,Salad,5.0,5.0,25.0,Cash,Takeaway,2023-11-07
48,TXN_6342161,Salad,5.0,5.0,25.0,Unknown,Takeaway,2023-01-08
8925,TXN_1373048,Salad,5.0,5.0,25.0,Unknown,In-Store,2023-08-31
8875,TXN_8740501,Salad,5.0,5.0,25.0,Unknown,Unknown,2023-02-05
8868,TXN_3550177,Salad,5.0,5.0,25.0,Cash,Unknown,2023-09-13
8859,TXN_1749930,Salad,5.0,5.0,25.0,Unknown,In-Store,2023-07-01
8758,TXN_4346813,Salad,5.0,5.0,25.0,Unknown,Unknown,2023-07-07
8751,TXN_7172693,Unknown,5.0,5.0,25.0,Credit Card,In-Store,2023-03-23
8737,TXN_4127927,Salad,5.0,5.0,25.0,Unknown,Takeaway,2023-08-04


In [29]:
col = 'Total Spent'
q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
iqr = q3 - q1
df[col] = df[col].clip(lower=q1 - 1.5 * iqr, upper=q3 + 1.5 * iqr)

In [30]:
col = 'Total Spent'
q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
iqr = q3 - q1
rows_before = len(df)
df = df[(df[col] >= q1 - 1.5 * iqr) & (df[col] <= q3 + 1.5 * iqr)]
print("Rows removed:", rows_before - len(df))

Rows removed: 0


## Outlier Detection

Outliers in the numeric columns were identified with the IQR method: a 
value is flagged if it lies below Q1 − 1.5×IQR or above Q3 + 1.5×IQR. 
The IQR method was chosen because it does not assume a normal 
distribution.

| Column | Lower bound | Upper bound | Outliers | Decision |
|---|---|---|---|---|
| Quantity | [x] | [x] | [N] | [Retain] |
| Price Per Unit | [x] | [x] | [N] | [Retain] |
| Total Spent | [x] | [x] | [N] | [Retain] |

**Decisions:** [Quantity and Price Per Unit: values are within a 
realistic range for a café (prices come from a fixed menu, quantities 
are small), so they were retained. Total Spent: the flagged values 
equal Quantity × Price Per Unit for the highest-priced items and large 
orders, so they are genuine transactions and were retained. Removing 
them would bias revenue analysis downwards.]